In [1]:
#yolo_here
from ultralytics import YOLO
import torch
import numpy as np
import time

In [4]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.1+cu126
True
NVIDIA GeForce RTX 4070 Ti SUPER


In [2]:
model = YOLO("yolo26s.pt")

In [ ]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from ultralytics.utils.downloads import download


def visdrone2yolo(root: Path, split: str):
    root = Path(root)

    img_dir = root / split / "images"
    ann_dir = root / split / "annotations"
    label_dir = root / split / "labels"
    label_dir.mkdir(parents=True, exist_ok=True)

    def convert_box(size, box):
        dw = 1.0 / size[0]
        dh = 1.0 / size[1]
        return (
            (box[0] + box[2] / 2) * dw,
            (box[1] + box[3] / 2) * dh,
            box[2] * dw,
            box[3] * dh
        )

    for ann_file in tqdm(list(ann_dir.glob("*.txt")), desc=f"Converting {split}"):

        img_file = img_dir / ann_file.name.replace(".txt", ".jpg")

        if not img_file.exists():
            continue

        w, h = Image.open(img_file).size
        lines = []

        with open(ann_file, "r") as f:
            for row in f.read().strip().splitlines():
                row = row.split(",")

                if row[4] == "0":
                    continue

                cls = int(row[5]) - 1
                box = convert_box((w, h), tuple(map(int, row[:4])))

                lines.append(f"{cls} {box[0]} {box[1]} {box[2]} {box[3]}\n")

        with open(label_dir / ann_file.name, "w") as f:
            f.writelines(lines)




root = Path(r"D:/cv/Dataset")

urls = [
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip",
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip",
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-test-dev.zip",
]

download(urls, dir=root, curl=True, threads=4)

for split in [
    "VisDrone2019-DET-train",
    "VisDrone2019-DET-val",
    "VisDrone2019-DET-test-dev"
]:
    visdrone2yolo(root, split)

Unzipping D:\cv\Dataset\VisDrone2019-DET-val.zip to D:\cv\Dataset\VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.9Kfiles/s 0.6s.1ss
Unzipping D:\cv\Dataset\VisDrone2019-DET-test-dev.zip to D:\cv\Dataset\VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.9Kfiles/s 1.7s0.1s


In [4]:
start_time = time.time()
results = model.train(
    data= "D:\cv\Dataset\VisDrone.yaml",
    epochs=450,
    imgsz=1024,
    batch=8,
    patience=50,
    cos_lr=True,
    max_det=500
    )
end_time = time.time()

<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\Ray\AppData\Local\Temp\ipykernel_18576\118502689.py:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  data= "D:\cv\Dataset\VisDrone.yaml",


New https://pypi.org/project/ultralytics/8.4.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\cv\Dataset\VisDrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=450, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=500, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0

In [5]:

metrics = model.val(
    data=r"D:\cv\Dataset\VisDrone.yaml",
    split="val",
    plots=True,
    save_json=True
)


Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
YOLO26s summary (fused): 122 layers, 9,469,050 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1285.4422.8 MB/s, size: 126.7 KB)
val: Scanning D:\cv\Dataset\VisDrone2019-DET-val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 229.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 5.4it/s 6.4s0.1s
                   all        548      38759      0.622      0.481      0.494      0.303
            pedestrian        520       8844      0.689      0.568      0.607      0.302
                people        482       5125      0.645       0.42      0.458      0.192
               bicycle        364       1287      0.492      0.288      0.284      0.134
                   car        515      14064      0.813      0.834      0.854      0.615
        

In [6]:
cm = metrics.confusion_matrix.matrix

FP = cm[:-1, -1].sum()
FN = cm[-1, :-1].sum()

TP = np.diag(cm[:-1, :-1]).sum()
training_time = end_time - start_time
print(training_time)
print(f"mAP@0.5       : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")
print("TP =", int(TP))
print("FP =", int(FP))
print("FN =", int(FN))


36433.77910065651
mAP@0.5       : 0.4945
mAP@0.5:0.95  : 0.3034
Precision     : 0.6224
Recall        : 0.4806
TP = 23920
FP = 6582
FN = 11567


In [7]:
model_best = YOLO(r"D:\cv\models\yolo\runs\train-4\weights\best.pt")

In [8]:
test_metrics = model_best.val(
    data=r"D:\cv\Dataset\VisDrone.yaml",
    split="test",
    plots=True,
    save_json=True
)

Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
YOLO26s summary (fused): 122 layers, 9,469,050 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1669.9214.2 MB/s, size: 134.6 KB)
val: Scanning D:\cv\Dataset\VisDrone2019-DET-test-dev\labels.cache... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 375.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 9.5it/s 10.7s<0.1s
                   all       1610      75102      0.453      0.352      0.319      0.178
            pedestrian       1197      21006      0.486      0.322      0.302      0.116
                people        797       6376      0.456      0.172      0.166     0.0568
               bicycle        377       1302      0.248      0.126      0.101     0.0402
                   car       1530      28074      0.658      0.751      0.722      0.

In [9]:
cm = test_metrics.confusion_matrix.matrix

FP = cm[:-1, -1].sum()
FN = cm[-1, :-1].sum()

TP = np.diag(cm[:-1, :-1]).sum()


print(f"mAP@0.5       : {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {test_metrics.box.map:.4f}")
print(f"Precision     : {test_metrics.box.mp:.4f}")
print(f"Recall        : {test_metrics.box.mr:.4f}")
print("TP =", int(TP))
print("FP =", int(FP))
print("FN =", int(FN))

mAP@0.5       : 0.3190
mAP@0.5:0.95  : 0.1783
Precision     : 0.4528
Recall        : 0.3519
TP = 32735
FP = 10656
FN = 36687


mAP@0.5       : 0.4003
mAP@0.5:0.95  : 0.2360
Precision     : 0.5345
Recall        : 0.4236
TP = 40375
FP = 12370
FN = 28516
t22

mAP@0.5       : 0.3858
mAP@0.5:0.95  : 0.2242
Precision     : 0.5072
Recall        : 0.4137
TP = 38585
FP = 11669
FN = 30332
t20

mAP@0.5       : 0.3585
mAP@0.5:0.95  : 0.2050
Precision     : 0.4798
Recall        : 0.3872
TP = 35677
FP = 12818
FN = 33327
training_time=4335s
t12

mAP@0.5       : 0.2939
mAP@0.5:0.95  : 0.1655
Precision     : 0.4423
Recall        : 0.3268
TP = 31762
FP = 10871
FN = 37574
167m
t9

mAP@0.5       : 0.2727
mAP@0.5:0.95  : 0.1498
Precision     : 0.3984
Recall        : 0.3170
TP = 28927
FP = 12442
FN = 40638
t8

mAP@0.5       : 0.3909
mAP@0.5:0.95  : 0.2270
Precision     : 0.5141
Recall        : 0.4141
TP = 38133
FP = 11933
FN = 30817
t6

mAP@0.5       : 0.3190
mAP@0.5:0.95  : 0.1783
Precision     : 0.4528
Recall        : 0.3519
TP = 32735
FP = 10656
FN = 36687
t4

